# MVP — Previsão de Preço da PETR4 (B3)**Disciplina:** Machine Learning & Analytics  **Tarefa:** Prever o próximo preço de fechamento diário (**t+1**) da ação **PETR4** usando dados históricos.  **Dataset (URL):** `https://github.com/Guisteim/MVP_PUC/raw/refs/heads/MVP_Machine_Learn/bovespa_stocks_PETR4.csv`> Este notebook segue as boas práticas vistas em aula, com baseline, preparação de dados sem vazamento, validação temporal (TimeSeriesSplit), comparação de modelos e otimização de hiperparâmetros.

## Checklist — Definição do Problema- **Descrição:** previsão do preço de fechamento da PETR4 no dia **t+1** (regressão, série temporal).  - **Tipo:** aprendizado supervisionado (regressão / forecasting de curto prazo).  - **Premissas/Hipóteses:**   - A dinâmica intrínseca de preços depende de valores passados (lags/rolling).    - Efeitos de calendário (dia da semana/mês) podem capturar sazonalidade fraca.  - **Restrições:** usar somente informações disponíveis até **t** (evitar vazamento).  - **Atributos (esperados):** Data/Date, Open, High, Low, Close, Volume, Symbol (poderá variar conforme o CSV).

## 1) Setup e Importações

In [ ]:
# Reprodutibilidade e bibliotecasimport os, sys, math, json, warningswarnings.filterwarnings("ignore")import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom sklearn.model_selection import TimeSeriesSplit, GridSearchCVfrom sklearn.pipeline import Pipelinefrom sklearn.preprocessing import StandardScalerfrom sklearn.metrics import mean_absolute_error, mean_squared_errorfrom sklearn.linear_model import Ridgefrom sklearn.ensemble import RandomForestRegressor# Tentativa de usar XGBoost (opcional). Se não estiver instalado, instala.try:    import xgboost as xgb    HAS_XGB = Trueexcept Exception:    HAS_XGB = False    try:        import subprocess, sys        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])        import xgboost as xgb        HAS_XGB = True    except Exception:        HAS_XGB = FalseSEED = 42np.random.seed(SEED)

## 2) Carga dos Dados (via URL)

In [ ]:
URL = "https://github.com/Guisteim/MVP_PUC/raw/refs/heads/MVP_Machine_Learn/bovespa_stocks_PETR4.csv"# Carrega CSV e tenta padronizar nomes de colunas de forma robustadf = pd.read_csv(URL)# Normaliza nomesdf.columns = [c.strip().replace(" ", "_") for c in df.columns]lower_map = {c.lower(): c for c in df.columns}# Função auxiliar para pegar coluna por nomes alternativosdef pick(colnames):    for name in colnames:        if name in df.columns:            return name        if name.lower() in lower_map:            return lower_map[name.lower()]    return Nonedate_col = pick(["Date","Data","date","data"])close_col = pick(["Close","Adj_Close","close","fechamento"])open_col  = pick(["Open","open","abertura"])high_col  = pick(["High","high","max"])low_col   = pick(["Low","low","min"])vol_col   = pick(["Volume","Vol","volume"])sym_col   = pick(["Symbol","Ticker","symbol","ticker"])print("Colunas detectadas:")print({"Date": date_col, "Open": open_col, "High": high_col, "Low": low_col, "Close": close_col, "Volume": vol_col, "Symbol": sym_col})# Conversão de datas (robusta a timezone)if date_col is None:    raise ValueError("Coluna de data não encontrada. Verifique o CSV.")# Tenta diferentes estratégias de parsingdt = pd.to_datetime(df[date_col], errors="coerce", utc=True, infer_datetime_format=True)if dt.isna().mean() > 0.5:    # fallback para formato ISO misto    dt = pd.to_datetime(df[date_col], errors="coerce", format="ISO8601", utc=True)df["_Date"] = dt# Converte para horário local SP e remove timezone para facilitar modelagemtry:    df["_Date"] = df["_Date"].dt.tz_convert("America/Sao_Paulo").dt.tz_localize(None)except Exception:    df["_Date"] = df["_Date"].dt.tz_localize(None)# Mantém somente as colunas relevantes disponíveiskeep = ["_Date"]for c in [open_col, high_col, low_col, close_col, vol_col, sym_col]:    if c is not None and c in df.columns:        keep.append(c)df = df[keep].dropna().sort_values("_Date").reset_index(drop=True)df.rename(columns={"_Date": "Date", close_col: "Close", open_col: "Open", high_col: "High", low_col: "Low", vol_col: "Volume"}, inplace=True)print("\nAmostra:")display(df.head(10))print("\nShape:", df.shape)

## 3) Análise Exploratória (breve e objetiva)

In [ ]:
print("Tipos:")print(df.dtypes)print("\nValores ausentes (%):")print(df.isna().mean().round(4) * 100)# Gráfico simples do preço de fechamentoplt.figure(figsize=(10,4))plt.plot(df["Date"], df["Close"])plt.title("PETR4 — Fechamento ao longo do tempo")plt.xlabel("Data")plt.ylabel("Preço de Fechamento")plt.show()

> **Observações da EDA (preencha após rodar):**> - Tendência geral e volatilidade aparente.  > - Eventuais quebras estruturais (ex.: períodos de crise).  > - Presença/ausência de outliers e dados faltantes.

## 4) Feature Engineering (sem vazamento)

In [ ]:
df_feat = df.copy()# Cria retornos e log-retornosdf_feat["ret"] = df_feat["Close"].pct_change()df_feat["logret"] = np.log1p(df_feat["ret"])# Lags do preço/retornos (somente passado)for k in [1,2,3,5,10,20]:    df_feat[f"close_lag{k}"] = df_feat["Close"].shift(k)    df_feat[f"ret_lag{k}"] = df_feat["ret"].shift(k)# Médias móveis e volatilidade (rolling) — só usa passadofor w in [5,10,20]:    df_feat[f"ma{w}"] = df_feat["Close"].rolling(w).mean()    df_feat[f"std{w}"] = df_feat["Close"].rolling(w).std()# Efeitos de calendáriodf_feat["dow"] = df_feat["Date"].dt.weekday  # 0=segundadf_feat["month"] = df_feat["Date"].dt.month# TARGET: prever Close (t+1)df_feat["y"] = df_feat["Close"].shift(-1)# Remove linhas com NaN por causa de lags/rolling/shiftdf_feat = df_feat.dropna().reset_index(drop=True)print("Colunas finais:", df_feat.columns.tolist())display(df_feat.head(5))

> **Nota:** Toda transformação usa apenas informações **até t**, e o alvo é **Close (t+1)**. Assim evitamos **vazamento de dados**.

## 5) Divisão Treino/Teste e Validação Temporal

In [ ]:
# Ordena por data por garantiadf_feat = df_feat.sort_values("Date").reset_index(drop=True)# Separa X, ytarget = "y"cols_drop = ["Date", "y"]X = df_feat.drop(columns=cols_drop, errors="ignore")y = df_feat[target].values# Split: holdout final = últimos 20% para testen = len(df_feat)test_size = int(0.2 * n)X_train, X_test = X.iloc[:-test_size], X.iloc[-test_size:]y_train, y_test = y[:-test_size], y[-test_size:]print("Tamanhos:", len(X_train), len(X_test))

## 6) Baselines (Naive e Média Móvel)

In [ ]:
# Baseline 1: Naive (y_{t+1} = Close_t)naive_pred_test = df_feat["Close"].shift(1).iloc[-test_size:].valuesmae_naive = mean_absolute_error(y_test, naive_pred_test)rmse_naive = mean_squared_error(y_test, naive_pred_test, squared=False)# Baseline 2: Média móvel de 5 dias (prevê t+1 como MA5_t)ma5 = df_feat["Close"].rolling(5).mean().shift(1)  # MA até tma5_pred_test = ma5.iloc[-test_size:].valuesmae_ma5 = mean_absolute_error(y_test, ma5_pred_test)rmse_ma5 = mean_squared_error(y_test, ma5_pred_test, squared=False)print(f"Baseline Naive — MAE: {mae_naive:.4f} | RMSE: {rmse_naive:.4f}")print(f"Baseline MA5   — MAE: {mae_ma5:.4f} | RMSE: {rmse_ma5:.4f}")

## 7) Modelagem, Cross-Validation temporal e Otimização de Hiperparâmetros

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)results = []# 7.1) Ridge (com padronização)pipe_ridge = Pipeline([    ("scaler", StandardScaler()),    ("model", Ridge(random_state=SEED))])param_ridge = {    "model__alpha": [0.01, 0.1, 1.0, 10.0, 50.0]}gcv_ridge = GridSearchCV(pipe_ridge, param_grid=param_ridge, scoring="neg_mean_absolute_error",                         cv=tscv, n_jobs=-1, refit=True)gcv_ridge.fit(X_train, y_train)y_pred_ridge = gcv_ridge.predict(X_test)results.append(("Ridge", gcv_ridge.best_params_, mean_absolute_error(y_test, y_pred_ridge),                mean_squared_error(y_test, y_pred_ridge, squared=False)))# 7.2) RandomForestrf = RandomForestRegressor(random_state=SEED, n_estimators=400, max_depth=None, min_samples_leaf=2, n_jobs=-1)param_rf = {    "n_estimators": [300, 500],    "max_depth": [None, 6, 10],    "min_samples_leaf": [1, 2, 4]}gcv_rf = GridSearchCV(rf, param_grid=param_rf, scoring="neg_mean_absolute_error",                      cv=tscv, n_jobs=-1, refit=True)gcv_rf.fit(X_train, y_train)y_pred_rf = gcv_rf.predict(X_test)results.append(("RandomForest", gcv_rf.best_params_, mean_absolute_error(y_test, y_pred_rf),                mean_squared_error(y_test, y_pred_rf, squared=False)))# 7.3) XGBoost (se disponível)if HAS_XGB:    xgbr = xgb.XGBRegressor(        random_state=SEED,        objective="reg:squarederror",        n_estimators=600,        learning_rate=0.05,        subsample=0.8,        colsample_bytree=0.8,        max_depth=6,        n_jobs=-1,    )    param_xgb = {        "n_estimators": [400, 800],        "max_depth": [4, 6, 8],        "learning_rate": [0.03, 0.05, 0.1],        "subsample": [0.7, 0.9],        "colsample_bytree": [0.7, 0.9]    }    gcv_xgb = GridSearchCV(xgbr, param_grid=param_xgb, scoring="neg_mean_absolute_error",                           cv=tscv, n_jobs=-1, refit=True)    gcv_xgb.fit(X_train, y_train)    y_pred_xgb = gcv_xgb.predict(X_test)    results.append(("XGBoost", gcv_xgb.best_params_, mean_absolute_error(y_test, y_pred_xgb),                    mean_squared_error(y_test, y_pred_xgb, squared=False)))else:    print("XGBoost indisponível — pulando este modelo.")# Tabela de resultadosres_df = pd.DataFrame(results, columns=["Modelo","BestParams","MAE","RMSE"]).sort_values("MAE").reset_index(drop=True)display(res_df)

## 8) Escolha do Melhor Modelo e Avaliação no Teste

In [ ]:
# Decide pelo menor MAE (poderia ser RMSE/MAPE conforme preferência)best_row = res_df.iloc[0]best_name = best_row["Modelo"]print("Melhor por MAE:", best_name)# Recupera o estimador treinado ideal para previsões e análiseif best_name == "Ridge":    best_est = gcv_ridge.best_estimator_    y_pred = gcv_ridge.predict(X_test)elif best_name == "RandomForest":    best_est = gcv_rf.best_estimator_    y_pred = gcv_rf.predict(X_test)else:    best_est = gcv_xgb.best_estimator_    y_pred = gcv_xgb.predict(X_test)mae = mean_absolute_error(y_test, y_pred)rmse = mean_squared_error(y_test, y_pred, squared=False)mape = np.mean(np.abs((y_test - y_pred) / np.maximum(1e-8, np.abs(y_test)))) * 100print(f"Teste — {best_name}:  MAE={mae:.4f} | RMSE={rmse:.4f} | MAPE={mape:.2f}%")print(f"Baselines → Naive MAE={mae_naive:.4f} / RMSE={rmse_naive:.4f}  |  MA5 MAE={mae_ma5:.4f} / RMSE={rmse_ma5:.4f}")# Gráfico: Real vs Preditodates_test = df_feat["Date"].iloc[-len(y_test):].valuesplt.figure(figsize=(12,5))plt.plot(dates_test, y_test, label="Real")plt.plot(dates_test, y_pred, label=f"Predito — {best_name}")plt.title("PETR4 — Real vs Predito no Conjunto de Teste")plt.xlabel("Data"); plt.ylabel("Preço de Fechamento (t+1)")plt.legend()plt.show()

## 9) Conclusões e Próximos Passos- **Modelo vencedor:** escolhido com base na menor **MAE** em teste (ver tabela e impressão acima).  - **Comparação com baselines:** o modelo deve **superar Naive e MA5** para justificar seu uso.  - **Limitações:**   - O problema de previsão de preço **um passo à frente** é ruidoso; melhorias incrementais são esperadas, não milagres.    - Fatores exógenos (notícias, macroeconomia, câmbio, petróleo) **não estão** no dataset e podem elevar erro.  - **Melhorias sugeridas:**   - Incluir **variáveis exógenas** (ex.: Brent, USD/BRL, Ibovespa futuro).    - Ajustar janelas de *lags* e *rollings* conforme otimização de hiperparâmetros.    - Avaliar **modelos específicos de séries temporais** (ARIMA, Prophet) e **arquiteturas profundas** (LSTM/Temporal CNN).    - Validar janelas **walk-forward** (treina → valida → avança) para maior realismo.

## 10) Boas Práticas e Reprodutibilidade- Seed fixada (`SEED=42`).  - Validação temporal (`TimeSeriesSplit`) para evitar vazamento.  - **Pipelines** foram utilizados (Ridge) e **grids** documentados.  - Código segmentado e textualização das decisões.  - Métricas: **MAE, RMSE e MAPE** reportadas.